In [147]:
from __future__ import annotations

import sys
import matplotlib

# --- Backend selection (EXE-friendly logic) ---
if getattr(sys, "frozen", False):
    matplotlib.use("Qt5Agg")
else:
    try:
        matplotlib.use("TkAgg")
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button

try:
    from astroquery.jplhorizons import Horizons
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("astroquery is required. Install with: pip install astroquery") from e

from astropy.time import Time

from config import *

from geometry import (
    ensure_utc,
    horizons_times_to_utc,
    angle_deg_vectorized
)

from horizons import *

from events import (
    find_conj_opp_events
)

from plotting import (
    _mask_to_spans,
    maximize_figure,
    refresh_legend,
    style_connector,
    make_readout_text,
    draw_guide_orbits,
    draw_sun,
    draw_planet_tracks,
    draw_current_markers,
    draw_events,
    draw_connectors,
    compute_view_lim,
    build_analysis_figure,
    set_view_limits,
    register_artists
    )

from ui import(apply_planet_visibility, 
               update,
               reset_view,
               key_press)



In [148]:
df = build_dataset()

C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 3012 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 3012 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 3012 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 3012 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-pa

In [149]:
print (df)

                             t       x_E       y_E       z_E       x_V  \
0    2025-07-01 00:00:00+00:00  0.160485 -1.003878  0.000059  0.688452   
1    2025-07-02 00:00:00+00:00  0.177173 -1.001079  0.000059  0.694523   
2    2025-07-03 00:00:00+00:00  0.193809 -0.997998  0.000059  0.700060   
3    2025-07-04 00:00:00+00:00  0.210389 -0.994636  0.000060  0.705059   
4    2025-07-05 00:00:00+00:00  0.226908 -0.990995  0.000060  0.709515   
...                        ...       ...       ...       ...       ...   
4287 2037-03-27 00:00:00+00:00 -0.991987 -0.106880  0.000015  0.714840   
4288 2037-03-28 00:00:00+00:00 -0.990271 -0.124028  0.000016  0.711138   
4289 2037-03-29 00:00:00+00:00 -0.988261 -0.141138  0.000016  0.706886   
4290 2037-03-30 00:00:00+00:00 -0.985960 -0.158205  0.000017  0.702087   
4291 2037-03-31 00:00:00+00:00 -0.983367 -0.175224  0.000018  0.696744   

           y_V       z_V       x_J       y_J       z_J  ...  \
0    -0.231416 -0.042903 -0.319118  5.135144 -0.

In [150]:
df["t"] = ensure_utc(df["t"])
MAX_IDX = len(df) - 1

# Default mark
mark_dt = pd.Timestamp(MARK, tz="UTC")
diffs = (df["t"] - mark_dt).dt.total_seconds().abs()
i0 = int(diffs.idxmin()) if diffs.notna().any() else len(df) // 2

dates = df["t"].dt.tz_convert("UTC")
days = dates.dt.tz_localize(None)  # Matplotlib-friendly naive datetime

In [151]:
#print (df["t"], MAX_IDX)
#print (diffs)
#print (i0)
print (dates)
print (days)



0      2025-07-01 00:00:00+00:00
1      2025-07-02 00:00:00+00:00
2      2025-07-03 00:00:00+00:00
3      2025-07-04 00:00:00+00:00
4      2025-07-05 00:00:00+00:00
                  ...           
4287   2037-03-27 00:00:00+00:00
4288   2037-03-28 00:00:00+00:00
4289   2037-03-29 00:00:00+00:00
4290   2037-03-30 00:00:00+00:00
4291   2037-03-31 00:00:00+00:00
Name: t, Length: 4292, dtype: datetime64[ns, UTC]
0      2025-07-01
1      2025-07-02
2      2025-07-03
3      2025-07-04
4      2025-07-05
          ...    
4287   2037-03-27
4288   2037-03-28
4289   2037-03-29
4290   2037-03-30
4291   2037-03-31
Name: t, Length: 4292, dtype: datetime64[ns]


In [152]:
v_conj, v_opp = find_conj_opp_events(df["venus_elong_deg"].to_numpy())
print (v_conj)

[189, 481, 772, 1066, 1362, 1650, 1938, 2232, 2528, 2820, 3109, 3400, 3691, 3986, 4282]


In [153]:
j_conj, j_opp = find_conj_opp_events(df["jup_elong_deg"].to_numpy())
print (j_conj)
print (j_opp)

[394, 791, 1188, 1583, 1979, 2375, 2774, 3175, 3577, 3980]
[193, 590, 986, 1381, 1777, 2175, 2575, 2977, 3380, 3782, 4183]


In [154]:
def draw_planet_tracks_TEST (ax, df, planet_enabled, planet_artists):

    ax.plot(df["x_E"], df["y_E"], color=EARTH_COLOR, lw=1.5, alpha=0.55, label="Earth (HORIZONS)")

    if planet_enabled["Venus"]:
        t = ax.plot(
            df["x_V"], 
            df["y_V"], 
            color=VENUS_COLOR, 
            lw=1.8, 
            alpha=0.50, 
            label="Venus (HORIZONS)")[0]
        register_artists(planet_artists,"Venus", t)

    if planet_enabled["Mars"]:
        t = ax.plot(
            df["x_M"], 
            df["y_M"], 
            color=MARS_COLOR,
            lw=1.8, alpha=0.50,
            label="Mars (HORIZONS)")[0]
        register_artists(planet_artists,"Mars", t)

    if planet_enabled["Jupiter"]:
        t = ax.plot(
            df["x_J"], 
            df["y_J"], 
            color=JUPITER_COLOR, 
            lw=2.0, 
            alpha=0.55, 
            label="Jupiter (HORIZONS)")[0]
        register_artists(planet_artists,"Jupiter", t)

    if planet_enabled["Saturn"]:
        t = ax.plot(
            df["x_S"], 
            df["y_S"], 
            color=SATURN_COLOR, 
            lw=2.0, 
            alpha=0.45, 
            label="Saturn (HORIZONS)")[0]
        register_artists(planet_artists,"Saturn", t)
        


    """    
    venus_track = ax.plot(df["x_V"], df["y_V"], color=VENUS_COLOR, lw=1.8, alpha=0.50, label="Venus (HORIZONS)")[0]
    mars_track = ax.plot(df["x_M"], df["y_M"], color=MARS_COLOR, lw=1.8, alpha=0.50, label="Mars (HORIZONS)")[0]
    jup_track  = ax.plot(df["x_J"], df["y_J"], color=JUPITER_COLOR, lw=2.0, alpha=0.55, label="Jupiter (HORIZONS)")[0]
    sat_track  = ax.plot(df["x_S"], df["y_S"], color=SATURN_COLOR, lw=2.0, alpha=0.45, label="Saturn (HORIZONS)")[0]


    register_artists(planet_artists,"Venus", venus_track)    
    register_artists(planet_artists,"Mars", mars_track)
    register_artists(planet_artists,"Jupiter", jup_track)
    register_artists(planet_artists,"Saturn", sat_track)

    """

In [155]:
def draw_current_markers_TEST(ax, df, i0, planet_enabled, planet_artists):
    E = ax.scatter([df["x_E"].iloc[i0]], [df["y_E"].iloc[i0]], s=EARTH_SIZE, color=EARTH_COLOR, edgecolor="white",
                    label="_nolegend_", zorder=6)

    if planet_enabled["Venus"]:
        planet_mark = ax.scatter(
            [df["x_V"].iloc[i0]], 
            [df["y_V"].iloc[i0]], 
            s=VENUS_SIZE, 
            color=VENUS_COLOR, 
            edgecolor="white",
            label="_nolegend_", 
            zorder=6)
        register_artists(planet_artists,"Venus", planet_mark)

    if planet_enabled["Mars"]:
        planet_mark = ax.scatter(
            [df["x_M"].iloc[i0]], 
            [df["y_M"].iloc[i0]], 
            s=MARS_SIZE, 
            color=MARS_COLOR, 
            edgecolor="white",
            label="_nolegend_", 
            zorder=6)
        register_artists(planet_artists,"Mars", planet_mark)

    if planet_enabled["Jupiter"]:
        planet_mark = ax.scatter(
            [df["x_J"].iloc[i0]], 
            [df["y_J"].iloc[i0]], 
            s=JUPITER_SIZE, 
            color=JUPITER_COLOR, 
            edgecolor="white",
            label="_nolegend_", 
            zorder=6)
        register_artists(planet_artists,"Jupiter", planet_mark)
        
    if planet_enabled["Saturn"]:
        planet_mark = ax.scatter(
            [df["x_S"].iloc[i0]], 
            [df["y_S"].iloc[i0]], 
            s=SATURN_SIZE, 
            color=SATURN_COLOR, 
            edgecolor="white",
            label="_nolegend_", 
            zorder=6)
        register_artists(planet_artists,"Saturn", planet_mark)

    """
    
    V = ax.scatter([df["x_V"].iloc[i0]], [df["y_V"].iloc[i0]], s=VENUS_SIZE, color=VENUS_COLOR, edgecolor="white",
                    label="_nolegend_", zorder=6)
    
    M = ax.scatter([df["x_M"].iloc[i0]], [df["y_M"].iloc[i0]], s=MARS_SIZE, color=MARS_COLOR, edgecolor="white",
                    label="_nolegend_", zorder=6)
    J = ax.scatter([df["x_J"].iloc[i0]], [df["y_J"].iloc[i0]], s=JUPITER_SIZE, color=JUPITER_COLOR, edgecolor="white",
                    label="_nolegend_", zorder=6)
    S = ax.scatter([df["x_S"].iloc[i0]], [df["y_S"].iloc[i0]], s=SATURN_SIZE, color=SATURN_COLOR, edgecolor="white",
                    label="_nolegend_", zorder=6)


    register_artists(planet_artists,"Venus", V)


    register_artists(planet_artists,"Mars", M)
    register_artists(planet_artists,"Jupiter", J)
    register_artists(planet_artists,"Saturn", S)
    
    return V, E, M, J, S

    """
    return E


In [156]:
def draw_connectors_TEST(ax, df, planet_enabled, planet_artists, i0):

    if planet_enabled["Venus"]:
        connector_line = ax.plot(
            [df["x_E"].iloc[i0], 
             df["x_V"].iloc[i0]], 
            [df["y_E"].iloc[i0], 
             df["y_V"].iloc[i0]],
            color="white", 
            alpha=0.55, 
            lw=CONNECTOR_LW_USABLE, zorder=5)[0]
        register_artists(planet_artists,"Venus", connector_line)

    if planet_enabled["Mars"]:
        connector_line = ax.plot(
            [df["x_E"].iloc[i0], 
             df["x_M"].iloc[i0]], 
            [df["y_E"].iloc[i0], 
             df["y_M"].iloc[i0]],
            color="white", 
            alpha=0.55, 
            lw=CONNECTOR_LW_USABLE, zorder=5)[0]
        register_artists(planet_artists,"Mars", connector_line)


    if planet_enabled["Jupiter"]:
        connector_line = ax.plot(
            [df["x_E"].iloc[i0], 
             df["x_J"].iloc[i0]], 
            [df["y_E"].iloc[i0], 
             df["y_J"].iloc[i0]],
            color="white", 
            alpha=0.65, 
            lw=CONNECTOR_LW_USABLE, zorder=5)[0]
        register_artists(planet_artists,"Jupiter", connector_line)

    if planet_enabled["Saturn"]:
        connector_line = ax.plot(
            [df["x_E"].iloc[i0], 
             df["x_S"].iloc[i0]], 
            [df["y_E"].iloc[i0], 
             df["y_S"].iloc[i0]],
            color="white", 
            alpha=0.45, 
            lw=CONNECTOR_LW_USABLE, zorder=5)[0]
        register_artists(planet_artists,"Saturn", connector_line)

    
"""    
    def draw_connectors(ax, df, planet_artists, i0):


    lineEV = ax.plot([df["x_E"].iloc[i0], df["x_V"].iloc[i0]], [df["y_E"].iloc[i0], df["y_V"].iloc[i0]],
                      color="white", alpha=0.55, lw=CONNECTOR_LW_USABLE, zorder=5)[0]


    lineEM = ax.plot([df["x_E"].iloc[i0], df["x_M"].iloc[i0]], [df["y_E"].iloc[i0], df["y_M"].iloc[i0]],
                      color="white", alpha=0.55, lw=CONNECTOR_LW_USABLE, zorder=5)[0]
    lineEJ = ax.plot([df["x_E"].iloc[i0], df["x_J"].iloc[i0]], [df["y_E"].iloc[i0], df["y_J"].iloc[i0]],
                      color="white", alpha=0.65, lw=CONNECTOR_LW_USABLE, zorder=5)[0]
    lineES = ax.plot([df["x_E"].iloc[i0], df["x_S"].iloc[i0]], [df["y_E"].iloc[i0], df["y_S"].iloc[i0]],
                      color="white", alpha=0.45, lw=CONNECTOR_LW_USABLE, zorder=5)[0]


    register_artists(planet_artists,"Venus", lineEV)
    
    register_artists(planet_artists,"Mars", lineEM)
    register_artists(planet_artists,"Jupiter", lineEJ)
    register_artists(planet_artists,"Saturn", lineES)

    return lineEV, lineEM, lineEJ, lineES
    
    """


'    \n    def draw_connectors(ax, df, planet_artists, i0):\n\n\n    lineEV = ax.plot([df["x_E"].iloc[i0], df["x_V"].iloc[i0]], [df["y_E"].iloc[i0], df["y_V"].iloc[i0]],\n                      color="white", alpha=0.55, lw=CONNECTOR_LW_USABLE, zorder=5)[0]\n\n\n    lineEM = ax.plot([df["x_E"].iloc[i0], df["x_M"].iloc[i0]], [df["y_E"].iloc[i0], df["y_M"].iloc[i0]],\n                      color="white", alpha=0.55, lw=CONNECTOR_LW_USABLE, zorder=5)[0]\n    lineEJ = ax.plot([df["x_E"].iloc[i0], df["x_J"].iloc[i0]], [df["y_E"].iloc[i0], df["y_J"].iloc[i0]],\n                      color="white", alpha=0.65, lw=CONNECTOR_LW_USABLE, zorder=5)[0]\n    lineES = ax.plot([df["x_E"].iloc[i0], df["x_S"].iloc[i0]], [df["y_E"].iloc[i0], df["y_S"].iloc[i0]],\n                      color="white", alpha=0.45, lw=CONNECTOR_LW_USABLE, zorder=5)[0]\n\n\n    register_artists(planet_artists,"Venus", lineEV)\n\n    register_artists(planet_artists,"Mars", lineEM)\n    register_artists(planet_artists,"Jupiter",

In [157]:
def draw_events_TEST(ax, df, planet_artists, planet: str, xcol: str, ycol: str, opp_idxs, conj_idxs, size=EVENT_SIZE, fs=8):
    # Opposition

    show_labels = False
    for k, ii in enumerate(opp_idxs):

    
        h = ax.scatter(
            df[xcol].iloc[ii], 
            df[ycol].iloc[ii],
            s=size, 
            facecolor="none", 
            edgecolor=EVENT_OPP_COLOR, 
            lw=EVENT_MARKER_LW, 
            zorder=9,
            label="Opposition" if (planet == "Jupiter" and k == 0) else "_nolegend_")
        register_artists(planet_artists,planet, h)

        if show_labels:
            #yoff = 0.18 if (k % 2 == 0) else 0.30
            yoff = -0.08
            t = ax.text(df[xcol].iloc[ii], 
                        df[ycol].iloc[ii] + yoff,
                        f"{df['t'].iloc[ii].strftime(DATE_FMT)}",
                        ha="center", 
                        va="bottom", 
                        fontsize=fs, 
                        color="white",
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.55, lw=0))
            
            register_artists(planet_artists,planet, t)
            
    # Conjunction
    for k, ii in enumerate(conj_idxs):
        h = ax.scatter(df[xcol].iloc[ii], df[ycol].iloc[ii],
                        s=size, facecolor="none", edgecolor=EVENT_CONJ_COLOR, lw=EVENT_MARKER_LW, zorder=9,
                        label="Conjunction" if (planet == "Jupiter" and k == 0) else "_nolegend_")
        register_artists(planet_artists,planet, h)
        
        if show_labels:
            #yoff = -0.22 if (k % 2 == 0) else -0.36
            yoff = 0.08
            t = ax.text(df[xcol].iloc[ii], df[ycol].iloc[ii] + yoff,
                            f"{df['t'].iloc[ii].strftime(DATE_FMT)}",
                            ha="center", va="top", fontsize=fs, color="white",
                            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.55, lw=0))
            register_artists(planet_artists,planet, t)


In [163]:
def style_connector_NEW(planet, line, elong_deg: float):
    if line is None:
        return
    if not np.isfinite(elong_deg):
        line.set_alpha(0.0)
        return

    print(planet)
    
    if planet == "Venus":
        if elong_deg < float(ELONG_HIDE_DEG_INNER_PLANET):
            line.set_alpha(0.0)
            line.set_linewidth(0.0)
            return
        if elong_deg < float(ELONG_DIFFICULT_DEG_INNER_PLANET):
            line.set_alpha(0.25); line.set_color("crimson"); line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_DIFFICULT)
        elif elong_deg < float(ELONG_USABLE_DEG_INNER_PLANET):
            line.set_alpha(0.35); line.set_color("orange");  line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_MARGINAL)
        elif elong_deg < float(ELONG_EXCELLENT_DEG_INNER_PLANET):
            line.set_alpha(0.55); line.set_color("yellowgreen"); line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_USABLE)
        else:
            line.set_alpha(0.75); line.set_color("lime"); line.set_linestyle("solid"); line.set_linewidth(CONNECTOR_LW_EXCELLENT)

    if planet == "Mars" or planet == "Jupiter" or planet == "Saturn":
        if elong_deg < float(ELONG_HIDE_DEG_OUTER_PLANET):
            line.set_alpha(0.0)
            line.set_linewidth(0.0)
            return
        
        if elong_deg < float(ELONG_DIFFICULT_DEG_OUTER_PLANET):
            line.set_alpha(0.25); line.set_color("crimson"); line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_DIFFICULT)
        elif elong_deg < float(ELONG_USABLE_DEG_OUTER_PLANET):
            line.set_alpha(0.35); line.set_color("orange");  line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_MARGINAL)
        elif elong_deg < float(ELONG_EXCELLENT_DEG_OUTER_PLANET):
            line.set_alpha(0.55); line.set_color("yellowgreen"); line.set_linestyle((0, (1, 3))); line.set_linewidth(CONNECTOR_LW_USABLE)
        else:
            line.set_alpha(0.75); line.set_color("lime"); line.set_linestyle("solid"); line.set_linewidth(CONNECTOR_LW_EXCELLENT)


"""
SHOW_SOLAR_GLARE_MASK = True

ELONG_HIDE_DEG_OUTER_PLANET = 10.0
ELONG_DIFFICULT_DEG_OUTER_PLANET = 15.0
ELONG_USABLE_DEG_OUTER_PLANET = 25.0
ELONG_EXCELLENT_DEG_OUTER_PLANET = 60.0

ELONG_HIDE_DEG_INNER_PLANET = 10.0
ELONG_DIFFICULT_DEG_INNER_PLANET = 15.0
ELONG_USABLE_DEG_INNER_PLANET = 25.0
ELONG_EXCELLENT_DEG_INNER_PLANET = 40.0

GLARE_SHADE_BELOW_DEG = ELONG_DIFFICULT_DEG_OUTER_PLANET #ELONG_DIFFICULT_DEG

GLARE_SHADE_COLOR = "crimson"
GLARE_SHADE_ALPHA = 0.10
"""

'\nSHOW_SOLAR_GLARE_MASK = True\n\nELONG_HIDE_DEG_OUTER_PLANET = 10.0\nELONG_DIFFICULT_DEG_OUTER_PLANET = 15.0\nELONG_USABLE_DEG_OUTER_PLANET = 25.0\nELONG_EXCELLENT_DEG_OUTER_PLANET = 60.0\n\nELONG_HIDE_DEG_INNER_PLANET = 10.0\nELONG_DIFFICULT_DEG_INNER_PLANET = 15.0\nELONG_USABLE_DEG_INNER_PLANET = 25.0\nELONG_EXCELLENT_DEG_INNER_PLANET = 40.0\n\nGLARE_SHADE_BELOW_DEG = ELONG_DIFFICULT_DEG_OUTER_PLANET #ELONG_DIFFICULT_DEG\n\nGLARE_SHADE_COLOR = "crimson"\nGLARE_SHADE_ALPHA = 0.10\n'

In [164]:
def update_TEST(
    val,
    MAX_IDX,
    df,
    dates,
    days,
    txt,
    fig1,
    ax1,
    blue_dot_marker,
    planet_enabled,
    planet_artists
):
    
    i = int(val)
    i = max(0, min(MAX_IDX, i))
    
    # Markers
    blue_dot_marker.set_offsets([[df["x_E"].iloc[i], df["y_E"].iloc[i]]])

    #redraw marker and connectors (inc style) for enabled planets only
    # markers are index 2 in  planet artists
    # connectors are are index 3 in planet artists

    if planet_enabled["Venus"]:
        planet_artists["Venus"][2].set_offsets([[df["x_V"].iloc[i], df["y_V"].iloc[i]]])
        planet_artists["Venus"][3].set_data([df["x_E"].iloc[i], df["x_V"].iloc[i]], [df["y_E"].iloc[i], df["y_V"].iloc[i]])
        style_connector_NEW("Venus", planet_artists["Venus"][3], float(df["venus_elong_deg"].iloc[i]))
        
    if planet_enabled["Mars"]:
        planet_artists["Mars"][2].set_offsets([[df["x_M"].iloc[i], df["y_M"].iloc[i]]])
        planet_artists["Mars"][3].set_data([df["x_E"].iloc[i], df["x_M"].iloc[i]], [df["y_E"].iloc[i], df["y_M"].iloc[i]])
        style_connector_NEW("Mars", planet_artists["Mars"][3], float(df["mars_elong_deg"].iloc[i]))
 
    if planet_enabled["Jupiter"]:
        planet_artists["Jupiter"][2].set_offsets([[df["x_J"].iloc[i], df["y_J"].iloc[i]]])
        planet_artists["Jupiter"][3].set_data([df["x_E"].iloc[i], df["x_J"].iloc[i]], [df["y_E"].iloc[i], df["y_J"].iloc[i]])
        style_connector_NEW("Jupiter", planet_artists["Jupiter"][3], float(df["jup_elong_deg"].iloc[i])) 
        
    if planet_enabled["Saturn"]:
        planet_artists["Saturn"][2].set_offsets([[df["x_S"].iloc[i], df["y_S"].iloc[i]]])
        planet_artists["Saturn"][3].set_data([df["x_E"].iloc[i], df["x_S"].iloc[i]], [df["y_E"].iloc[i], df["y_S"].iloc[i]])
        style_connector_NEW("Saturn", planet_artists["Saturn"][3], float(df["sat_elong_deg"].iloc[i]))

    # Readout + view limits + legend
    txt.set_text(make_readout_text(i, dates, planet_enabled, df))
   # apply_planet_visibility(planet_artists, planet_enabled, ax1)
   # set_view_limits(ax1,planet_enabled)


    
    fig1.canvas.draw_idle()    


    """
    V.set_offsets([[df["x_V"].iloc[i], df["y_V"].iloc[i]]])
    E.set_offsets([[df["x_E"].iloc[i], df["y_E"].iloc[i]]])
    M.set_offsets([[df["x_M"].iloc[i], df["y_M"].iloc[i]]])
    J.set_offsets([[df["x_J"].iloc[i], df["y_J"].iloc[i]]])
    S.set_offsets([[df["x_S"].iloc[i], df["y_S"].iloc[i]]])

    # Connectors

    lineEV.set_data([df["x_E"].iloc[i], df["x_V"].iloc[i]], [df["y_E"].iloc[i], df["y_V"].iloc[i]])

    lineEM.set_data([df["x_E"].iloc[i], df["x_M"].iloc[i]], [df["y_E"].iloc[i], df["y_M"].iloc[i]])
    lineEJ.set_data([df["x_E"].iloc[i], df["x_J"].iloc[i]], [df["y_E"].iloc[i], df["y_J"].iloc[i]])
    lineES.set_data([df["x_E"].iloc[i], df["x_S"].iloc[i]], [df["y_E"].iloc[i], df["y_S"].iloc[i]])

    if SHOW_SOLAR_GLARE_MASK:

        style_connector(lineEV, float(df["venus_elong_deg"].iloc[i]))

        style_connector(lineEJ, float(df["jup_elong_deg"].iloc[i]))
        style_connector(lineEM, float(df["mars_elong_deg"].iloc[i]))
        style_connector(lineES, float(df["sat_elong_deg"].iloc[i]))

    # Readout + view limits + legend
    txt.set_text(make_readout_text(i, dates, planet_enabled, df))
    apply_planet_visibility(planet_artists, planet_enabled, ax1)
    set_view_limits(ax1,planet_enabled)

    # Fig2 cursor
    vline.set_xdata([days.iloc[i], days.iloc[i]])

    fig1.canvas.draw_idle()
    fig2.canvas.draw_idle()
    """

In [165]:


df = build_dataset()
df["t"] = ensure_utc(df["t"])
MAX_IDX = len(df) - 1

# Default mark
mark_dt = pd.Timestamp(MARK, tz="UTC")
diffs = (df["t"] - mark_dt).dt.total_seconds().abs()
i0 = int(diffs.idxmin()) if diffs.notna().any() else len(df) // 2

dates = df["t"].dt.tz_convert("UTC")
days = dates.dt.tz_localize(None)  # Matplotlib-friendly naive datetime

# Events
v_conj, v_opp = find_conj_opp_events(df["venus_elong_deg"].to_numpy())
j_conj, j_opp = find_conj_opp_events(df["jup_elong_deg"].to_numpy())
m_conj, m_opp = find_conj_opp_events(df["mars_elong_deg"].to_numpy())
s_conj, s_opp = find_conj_opp_events(df["sat_elong_deg"].to_numpy())

# ---------- FIG 1 ----------
fig1, ax1 = plt.subplots()
#fig1, ax1 = plt.subplots(figsize=(12.5, 9.5)) #, num=f" — Top-down View")

#fig1, ax1 = plt.subplots(figsize=(12.5, 9.5), num=f"{APP_NAME} {APP_VERSION} — Top-down View")

fig1.patch.set_facecolor(BACKGROUND_COLOR)
ax1.set_facecolor(BACKGROUND_COLOR)

maximize_figure(fig1)

planet_artists = {"Venus": [],"Jupiter": [], "Mars": [], "Saturn": []}

"""
planet_enabled = {
    "Venus": bool(SHOW_PLANET_VENUS),
    "Jupiter": bool(SHOW_PLANET_JUPITER),
    "Mars": bool(SHOW_PLANET_MARS),
    "Saturn": bool(SHOW_PLANET_SATURN),
}
"""

planet_enabled = {
    "Venus": True,
    "Mars": True,
    "Jupiter": True,
    "Saturn": True,
}

refresh_legend(ax1)

theta = np.linspace(0, 2*np.pi, 1200)

# Guide orbits (only for enabled planets) # Index 0
draw_guide_orbits(ax1, theta, planet_enabled, planet_artists)  

# Sun
draw_sun(ax1)

# Tracks, Index 1
draw_planet_tracks_TEST(ax1, df, planet_enabled, planet_artists) 

# Current markers, Index 2
blue_dot_marker = draw_current_markers_TEST(ax1, df, i0, planet_enabled, planet_artists)

# Connectors, Index 3
draw_connectors_TEST(ax1, df, planet_enabled, planet_artists, i0)

# Draw events, Index (4.5)....

if planet_enabled["Venus"]:
    draw_events_TEST(ax1, df, planet_artists, "Venus",    "x_V", "y_V", v_opp, v_conj, size=EVENT_SIZE*0.85, fs=7.5)        

if planet_enabled["Mars"]:
    draw_events_TEST(ax1, df, planet_artists, "Mars",    "x_M", "y_M", m_opp, m_conj, size=EVENT_SIZE*0.85, fs=7.5)

if planet_enabled["Jupiter"]:
    draw_events_TEST(ax1, df, planet_artists, "Jupiter", "x_J", "y_J", j_opp, j_conj, size=EVENT_SIZE*0.85, fs=7.5)

if planet_enabled["Saturn"]:
    draw_events_TEST(ax1, df, planet_artists, "Saturn",  "x_S", "y_S", s_opp, s_conj, size=EVENT_SIZE*0.85, fs=7.5)  

ax1.set_aspect("equal")

#print ("----------ALL planet artists -----------------")
#print (planet_artists)

#print ("----------VENUS planet artists -----------------")
#print (planet_artists["Venus"][0])
#print (planet_artists["Venus"][1])
#print (planet_artists["Venus"][2])

ax1.set_xlabel("x (AU)", color="white")
ax1.set_ylabel("y (AU)", color="white")
ax1.tick_params(axis="x", colors="white")
ax1.tick_params(axis="y", colors="white")
ax1.set_title(f"Earth–Planet Geometry ({APP_VERSION}) — Heliocentric ecliptic", color="white")
ax1.grid(True, ls=":", alpha=0.35, color="white")

txt = ax1.text(
    0.98, 0.98, make_readout_text(i0, dates, planet_enabled, df),
    transform=ax1.transAxes, va="top", ha="right",
    fontsize=9, family="monospace", color="white",
    bbox=dict(boxstyle="round", facecolor="black", alpha=0.85, lw=0.5, edgecolor="white")
    )

print (txt)

slider_ax = fig1.add_axes([0.12, 0.06, 0.76, 0.03], facecolor="#222")
slider = Slider(slider_ax, "", 0, MAX_IDX, valinit=i0, valstep=1)
slider.valtext.set_color("white")

btn_ax = fig1.add_axes([0.90, 0.05, 0.08, 0.05])
btn = Button(btn_ax, "Reset")


slider.on_changed(
    lambda val: update_TEST(
        val,
        MAX_IDX,
        df,
        dates,
        days,
        txt,
        fig1,
        ax1,
        blue_dot_marker,
        planet_enabled,
        planet_artists          
    )
)

#txt = "Test text"

"""
update_TEST(
    0,
    MAX_IDX,
    df,
    dates,
    days,
    txt,
    fig1,
    ax1,
    blue_dot_marker,
    planet_enabled,
    planet_artists
)
"""

print ("end of code")

C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 3012 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 3012 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 3012 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 3012 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\luisd\AppData\Local\Programs\Python\Python313\Lib\site-pa

Text(0.98, 0.98, 'Date: 10-Jan-2026\nEarth–Venus:    1.711 AU   Elong:   1.1°\nEarth–Jupiter:  4.232 AU   Elong: 179.5°\nEarth–Mars:    2.403 AU   Elong:   1.0°\nEarth–Saturn:  9.856 AU   Elong:  67.0°\nHelio sep V: 177.3° J:  0.4° M: 178.4° S: 107.5°\nSun glare: Venus elong < 15°, Mars elong < 15° (hard/unobservable)')
end of code


In [42]:
for i, artist in enumerate(planet_artists["Venus"]):
    print(i, type(artist), artist)

0 <class 'matplotlib.lines.Line2D'> Line2D(Venus orbit (0.723 AU))
1 <class 'matplotlib.lines.Line2D'> Line2D(Venus (HORIZONS))
2 <class 'matplotlib.collections.PathCollection'> <matplotlib.collections.PathCollection object at 0x000001CB52B00E10>
3 <class 'matplotlib.lines.Line2D'> Line2D(_child14)
4 <class 'matplotlib.collections.PathCollection'> <matplotlib.collections.PathCollection object at 0x000001CB52B01590>
5 <class 'matplotlib.text.Text'> Text(0.3186869793935598, 0.5671992375900539, '08-Jan-2033')
6 <class 'matplotlib.collections.PathCollection'> <matplotlib.collections.PathCollection object at 0x000001CB52B01950>
7 <class 'matplotlib.text.Text'> Text(0.1873987168827752, -0.6228760727768193, '06-Jan-2026')
8 <class 'matplotlib.collections.PathCollection'> <matplotlib.collections.PathCollection object at 0x000001CB52B01D10>
9 <class 'matplotlib.text.Text'> Text(0.6153711323586761, 0.4605118846793967, '25-Oct-2026')
10 <class 'matplotlib.collections.PathCollection'> <matplotlib.

In [115]:
print (df)

                             t       x_E       y_E       z_E       x_V  \
0    2025-07-01 00:00:00+00:00  0.160485 -1.003878  0.000059  0.688452   
1    2025-07-02 00:00:00+00:00  0.177173 -1.001079  0.000059  0.694523   
2    2025-07-03 00:00:00+00:00  0.193809 -0.997998  0.000059  0.700060   
3    2025-07-04 00:00:00+00:00  0.210389 -0.994636  0.000060  0.705059   
4    2025-07-05 00:00:00+00:00  0.226908 -0.990995  0.000060  0.709515   
...                        ...       ...       ...       ...       ...   
4287 2037-03-27 00:00:00+00:00 -0.991987 -0.106880  0.000015  0.714840   
4288 2037-03-28 00:00:00+00:00 -0.990271 -0.124028  0.000016  0.711138   
4289 2037-03-29 00:00:00+00:00 -0.988261 -0.141138  0.000016  0.706886   
4290 2037-03-30 00:00:00+00:00 -0.985960 -0.158205  0.000017  0.702087   
4291 2037-03-31 00:00:00+00:00 -0.983367 -0.175224  0.000018  0.696744   

           y_V       z_V       x_J       y_J       z_J  ...  \
0    -0.231416 -0.042903 -0.319118  5.135144 -0.

In [133]:
ELONGATION_THRESHOLDS = {
    "Venus": {
        "hide": 10,
        "difficult": 15,
        "usable": 25,
        "excellent": 40,
    },

    "Mars": {
        "hide": 10,
        "difficult": 15,
        "usable": 25,
        "excellent": 60,
    },

    "Jupiter": {
        "hide": 10,
        "difficult": 15,
        "usable": 25,
        "excellent": 60,
    },

    "Saturn": {
        "hide": 10,
        "difficult": 15,
        "usable": 25,
        "excellent": 60,
    },
}

print (ELONGATION_THRESHOLDS["Mars"]["hide"])


10


In [137]:
planet = "Saturn"
difficulty_level = "usable"
print (ELONGATION_THRESHOLDS[planet][difficulty_level])



25
